# Build

In [ ]:
import os
import pandas as pd

In [ ]:
from glob import glob

In [ ]:
base_dir = "/kaggle/input/skin-cancer-mnist-ham10000"

image_paths_part1 = glob(os.path.join(base_dir, "HAM10000_images_part_1", "*.jpg"))
image_paths_part2 = glob(os.path.join(base_dir, "HAM10000_images_part_2", "*.jpg"))

all_image_paths = image_paths_part1 + image_paths_part2

print(f"Part 1 image count: {len(image_paths_part1)}")
print(f"Part 2 image count: {len(image_paths_part2)}")
print(f"Total image count: {len(all_image_paths)}")

In [ ]:
# mapping image id to full path

image_path_dict = {os.path.splitext(os.path.basename(x))[0]: x for x in all_image_paths}


In [ ]:
metadata_path = os.path.join(base_dir, "HAM10000_metadata.csv")
skin_df = pd.read_csv(metadata_path)

In [ ]:
skin_df["path"] = skin_df["image_id"].map(image_path_dict)

skin_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

lesion_type_dict = {
    "nv": "Melanocytic nevi (Benign Mole)",
    "mel": "Melanoma (Cancer)",
    "bkl": "Benign keratosis (Benign)",
    "bcc": "Basal cell carcinoma (Cancer)",
    "akiec": "Actinic keratosis (Pre-Cancer)",
    "vasc": "Vascular lesion (Benign)",
    "df": "Dermatofibroma (Benign)"
}

skin_df["cell_type"] = skin_df["dx"].map(lesion_type_dict)

In [ ]:
plt.figure(figsize=(12,6))
plt.title("Skin Lesion Distribution", fontsize=15)
sns.countplot(y="cell_type", data=skin_df, order=skin_df["cell_type"].value_counts().index)
plt.xlabel("Number of Images", fontsize=12)
plt.ylabel("Lesionn Type", fontsize=12)
plt.show()

In [ ]:
print(skin_df["cell_type"].value_counts())

Needed to be balanced

In [ ]:
from PIL import Image

# grid of 5 random img
fig, axes = plt.subplots(1, 5, figsize=(20,6))
sample_data = skin_df.sample(5)

for i, (index, row) in enumerate(sample_data.iterrows()):

    img = Image.open(row["path"])

    axes[i].imshow(img)
    axes[i].set_title(f"Diagnosis:\n{row['cell_type']}", fontsize=10)
    axes[i].axis("off")
plt.show()


In [ ]:
import numpy as np
from PIL import Image

# fixing images to 128x128
IMG_SIZE = 128

def load_and_resize_images(df):

    image_data = []
    
    for path in df['path']:
        # Open
        img = Image.open(path)
        
        img = img.resize((IMG_SIZE, IMG_SIZE))
        
        image_data.append(np.array(img))
    
    return np.array(image_data)

print("..starting to resize images..")

X = load_and_resize_images(skin_df)

print(f"Final Shape: {X.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
le.fit(skin_df['dx'])
print("Classes:", le.classes_)

encoded_labels = le.transform(skin_df['dx'])

Y = to_categorical(encoded_labels)

print(f"Final label shapes: {Y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# train test split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

print(f"Training Data Shape: {x_train.shape}")
print(f"Test Data Shape:     {x_test.shape}")

# **Starting to modeling phase:**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential() #initialize empty model

# input layer
model.add(Input(shape=(128,128,3)))

# layer1, 32 filters, scanning 3x3
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

# layer2, 64 filters, looking for complex shapes
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

# layer3, expert analysis, 128 filters, looking for specific cancer patterns
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

# decision-maker
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# output
model.add(Dense(7, activation='softmax'))

#optimizer: Adam, loss: categorical_crossentropy
model.compile(optimizer='Adam',
             loss='categorical_crossentropy',
             metrics=['accuracy'])

model.summary()

**Training:**

In [ ]:
print("training started..")

history = model.fit(
    x_train,y_train,
    epochs=25,
    batch_size=32,
    validation_data=(x_test, y_test),
    verbose=1
)
print("--training completed--")

*Droput will be tuned, test score is not high enough*

In [ ]:
# save model
model.save('skin_cancer_model_v1.keras') 
print("Model saved to disk")

# report graphs
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

**Simple test for prediction**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

classes = {
    0: 'Actinic keratosis (akiec)',
    1: 'Basal cell carcinoma (bcc)',
    2: 'Benign keratosis (bkl)',
    3: 'Dermatofibroma (df)',
    4: 'Melanoma (mel)',
    5: 'Melanocytic nevi (nv)',
    6: 'Vascular lesion (vasc)'
}

# picking random image from test set
random_index = np.random.randint(0, len(x_test))
test_image = x_test[random_index]
true_label_index = np.argmax(y_test[random_index]) # correct

# model predict
prediction_scores = model.predict(np.expand_dims(test_image, axis=0))
predicted_label_index = np.argmax(prediction_scores) # The highest score

# results
plt.imshow(test_image.astype('uint8')) # skin image
plt.axis('off')
plt.show()

print(f"Actual Diagnosis:    {classes[true_label_index]}")
print(f"Model Prediction:    {classes[predicted_label_index]}")

# confidence scoring
confidence = prediction_scores[0][predicted_label_index] * 100
print(f"Confidence Score:    {confidence:.2f}%")

if true_label_index == predicted_label_index:
    print("Correct")
else:
    print("Wrong")

# **Confusion matrix:**

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# get predictions
print("...analyzing test images...")
y_pred_probs = model.predict(x_test)

# convert from probability to disease_id
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# matrix:
cm = confusion_matrix(y_true_classes, y_pred_classes)

# plot:
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes.values(),
            yticklabels=classes.values())

plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
import os
from IPython.display import FileLink

print("files in working directory:")
print(os.listdir('/kaggle/working'))

# download link
FileLink(r'skin_cancer_model_v1.keras')